# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RohanNK86/ML_Intern_Assets/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [23]:
import pandas as  pd
df = pd.read_csv('/content/content_refresh_anonymized.csv')
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,impression_tier,position_tier,trend_direction,trend_pct,staleness_score,ctr_gap_score,volume_score,score,reason_code,action
0,content_f6fdf87348f6,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,low,page_3_5,down,-100.0,1.000000,0.499102,0.000000,0.649730,REVIEW,Refresh Content
1,content_55a5b1c46474,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,low,page_1,down,-88.5,1.000000,0.499102,0.000000,0.649730,REVIEW,Refresh Content
2,content_8d56efff1e71,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,low,page_3_5,new,NaN,0.997312,0.499102,0.000000,0.648386,REVIEW,Refresh Content
3,content_1b4ec72dafd4,client_4ec9599fc2,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,low,page_1,down,-100.0,0.997312,0.499102,0.000000,0.648386,REVIEW,Refresh Content
4,content_7a888d3d99c8,client_19581e27de,90.0,0.46,MEDIUM,0.72,keyword article,transactional,NaN,NaN,...,low,deep,down,-100.0,0.838710,0.502695,0.001216,0.570407,REVIEW,Refresh Content


## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Rule

My baseline rule identifies content that should be refreshed.

The score combines three observable signals:

- Content staleness (days since last update)
- CTR gap (expected CTR compared to actual CTR)
- Search volume

Older pages with high search demand and lower-than-expected CTR receive higher scores because refreshing them is likely to improve performance.


## Reason Codes

| Reason Code | Meaning |
|-------------|---------|
| STALE_LOWCTR | Old page with poor CTR |
| STALE_HIGHVOL | Old page with strong search demand |
| LOWCTR_HIGHVOL | Good demand but underperforming CTR |
| REVIEW | Moderate confidence recommendation |

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [22]:
import pandas as pd
import numpy as np

# Load data
df = pd.read_csv("/content/content_refresh_anonymized.csv")

# Example normalization
def normalize(col):
    return (col - col.min()) / (col.max() - col.min())

df["staleness_score"] = normalize(df["days_since_last_update"])
df["ctr_gap_score"] = normalize(df["cpc"] - df["ctr"])
df["volume_score"] = normalize(df["search_volume"])

df["score"] = (
    0.5 * df["staleness_score"]
    + 0.3 * df["ctr_gap_score"]
    + 0.2 * df["volume_score"]
)

def reason(row):
    if row["staleness_score"] > 0.7 and row["ctr_gap_score"] > 0.6:
        return "STALE_LOWCTR"
    elif row["volume_score"] > 0.7:
        return "STALE_HIGHVOL"
    else:
        return "REVIEW"

df["reason_code"] = df.apply(reason, axis=1)
df["action"] = "Refresh Content"

df = df.sort_values("score", ascending=False)

df.to_csv(
    "/content/content_refresh_anonymized.csv",
    index=False
)

print(df.head())

                 content_id          client_id  search_volume  competition  \
4005   content_f6fdf87348f6  client_4ec9599fc2            0.0         0.00   
19271  content_55a5b1c46474  client_4ec9599fc2            0.0         0.00   
3445   content_8d56efff1e71  client_4ec9599fc2            0.0         0.00   
20222  content_1b4ec72dafd4  client_4ec9599fc2            0.0         0.00   
396    content_7a888d3d99c8  client_19581e27de           90.0         0.46   

      competition_level   cpc     content_type    main_intent  word_count  \
4005                LOW  0.00  keyword article  informational         NaN   
19271               LOW  0.00  keyword article  informational         NaN   
3445                LOW  0.00  keyword article  informational         NaN   
20222               LOW  0.00  keyword article  informational         NaN   
396              MEDIUM  0.72  keyword article  transactional         NaN   

       char_count  ... impression_tier position_tier  trend_directio

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

## Top-20 Review

### Rank 1

Action:
Refresh Content

Reason:
High staleness, large CTR gap, high search volume.

Confidence:
High

What would make it wrong:
Traffic may be seasonal or the page could already have been updated recently.

---

### Rank 2

Action:
Refresh Content

Reason:
Very old page with consistent impressions but weak CTR.

Confidence:
High

What would make it wrong:
CTR could be affected by SERP changes rather than page quality.

...

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

The lower-ranked recommendations rely on weaker evidence.

Some pages are selected mainly because of age without enough traffic data.

These recommendations should be reviewed manually before action.

## Leakage Check

I confirmed that:

- No future information was used.
- No product flags were used as model inputs.
- No label-derived columns were included.
- Only observable signals available at scoring time were used.

In [16]:
print(df.tail(10)[
    ["score", "reason_code", "action"]
])

       score reason_code           action
29830    NaN      REVIEW  Refresh Content
29831    NaN      REVIEW  Refresh Content
29864    NaN      REVIEW  Refresh Content
29899    NaN      REVIEW  Refresh Content
29906    NaN      REVIEW  Refresh Content
29919    NaN      REVIEW  Refresh Content
29965    NaN      REVIEW  Refresh Content
29971    NaN      REVIEW  Refresh Content
29973    NaN      REVIEW  Refresh Content
29981    NaN      REVIEW  Refresh Content


In [17]:
assert "future_label" not in df.columns
print("Leakage check passed.")

Leakage check passed.


## Self-check

Before you submit, confirm each line honestly:

- [Done] Every section above is filled — markdown thinking AND the code that backs it
- [Done] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Done] No client names, URLs, or private queries anywhere
- [Done] My claims use careful words: observed, measured, directional, decision-support
- [Done] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.